# ETA előrejelzés NYC Yellow Taxi adatokon

## Lineáris regresszió és interpretálhatóság

Ebben a notebookban a `clean_data/2009` adatokból építünk ETA (Estimated Time of Arrival) modellt.
A célváltozó: `trip_duration_minutes` (az út időtartama percben).

A notebook célja egyszerre:
1. **jó baseline modell** készítése (`LinearRegression`),
2. **hiperparaméter-választás** bemutatása (`Ridge`/`Lasso` + `GridSearchCV`),
3. **értelmezhetőség** biztosítása (koefficiensek + permutation importance).

Referencia: Christoph Molnar – *Interpretable Machine Learning*
https://christophm.github.io/interpretable-ml-book/

## Elméleti háttér röviden

A lineáris regresszió modell alakja:

$$
\hat{y} = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + ... + \beta_p x_p
$$

ahol:
- $\hat{y}$: becsült ETA percben,
- x_i: bemeneti jellemzők (távolság, koordináták, időjellemzők),
- $\beta_i$: modellparaméterek (koefficiensek).

Interpretáció:
- a koefficiens előjele mutatja az irányt (növeli/csökkenti a becslést),
- nagysága a hatás erősségét mutatja (a többi változó rögzítése mellett).

In [10]:
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.dataset as ds
from IPython.display import display

from sklearn.inspection import permutation_importance
from sklearn.linear_model import Lasso, LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, KFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [11]:
# --- Konfiguráció ---
DATA_DIR = Path('clean_data/2009')
PARQUET_FILES = sorted(DATA_DIR.glob('*.parquet'))

if not PARQUET_FILES:
    raise FileNotFoundError(f'Nincsenek parquet fájlok itt: {DATA_DIR.resolve()}')

# Egy közös korlát: ennyi sort olvasunk be és legfeljebb ennyit használunk tanításhoz
USE_SAMPLE = True
MAX_ROWS = 2_000_000

BATCH_SIZE = 250_000

print(f'Fájlok száma: {len(PARQUET_FILES)}')

Fájlok száma: 12


## Miért batch-es beolvasás?

A teljes 2009-es állomány nagyon nagy lehet (több tízmillió sor), ezért nem célszerű egyszerre memóriába tölteni.
A `pyarrow.dataset` scanner batch-ekben olvas, így stabilabb és memóriahatékonyabb a futás.

In [12]:
dataset = ds.dataset([str(path) for path in PARQUET_FILES], format='parquet')
scanner = dataset.scanner(batch_size=BATCH_SIZE, use_threads=True)

chunks = []
rows_loaded = 0

for i, rb in enumerate(scanner.to_batches(), start=1):
    chunk = rb.to_pandas(split_blocks=True, self_destruct=True)
    chunks.append(chunk)
    rows_loaded += len(chunk)

    if i % 5 == 0:
        print(f'Batch {i}: rows_loaded={rows_loaded:,}')

    if USE_SAMPLE and rows_loaded >= MAX_ROWS:
        break

df = pd.concat(chunks, ignore_index=True)
if USE_SAMPLE and len(df) > MAX_ROWS:
    df = df.iloc[:MAX_ROWS]

print(f'Beolvasott sorok: {len(df):,}')
display(df.head())

Batch 5: rows_loaded=1,000,000
Batch 10: rows_loaded=2,000,000
Beolvasott sorok: 2,000,000


,pickup_at,dropoff_at,passenger_count,trip_distance,pickup_lon,pickup_lat,dropoff_lon,dropoff_lat
0,2009-01-04 02:52:00,2009-01-04 03:02:00,1,2.63,-73.991957,40.721567,-73.993803,40.695922
1,2009-01-04 03:31:00,2009-01-04 03:38:00,3,4.55,-73.982102,40.736290,-73.955850,40.768030
2,2009-01-03 15:43:00,2009-01-03 15:57:00,5,10.35,-74.002587,40.739748,-73.869983,40.770225
3,2009-01-01 20:52:58,2009-01-01 21:14:00,1,5.00,-73.974267,40.790955,-73.996558,40.731849
4,2009-01-24 16:18:23,2009-01-24 16:24:56,1,0.40,-74.001580,40.719382,-74.008378,40.720350


## Célváltozó és feature engineering

A célváltozót időbélyegekből számoljuk:

`trip_duration_minutes = (dropoff_at - pickup_at)` percben.

Szűrések:
- hiányzó időpontok törlése,
- irreális időtartamok szűrése (`1..300` perc).

Új időfeature-ök:
- indulás órája (`pickup_hour`),
- hét napja (`pickup_weekday`),
- hónap (`pickup_month`).

In [13]:
df['pickup_at'] = pd.to_datetime(df['pickup_at'], errors='coerce')
df['dropoff_at'] = pd.to_datetime(df['dropoff_at'], errors='coerce')
df['trip_duration_minutes'] = (df['dropoff_at'] - df['pickup_at']).dt.total_seconds() / 60.0

df = df.dropna(subset=['pickup_at', 'dropoff_at', 'trip_duration_minutes'])
df = df[df['trip_duration_minutes'].between(1, 300)]

df['pickup_hour'] = df['pickup_at'].dt.hour
df['pickup_weekday'] = df['pickup_at'].dt.weekday
df['pickup_month'] = df['pickup_at'].dt.month

feature_cols = [
    'passenger_count', 'trip_distance',
    'pickup_lon', 'pickup_lat', 'dropoff_lon', 'dropoff_lat',
    'pickup_hour', 'pickup_weekday', 'pickup_month',
]

for col in feature_cols + ['trip_duration_minutes']:
    df[col] = pd.to_numeric(df[col], errors='coerce').astype('float32')

model_df = df.loc[:, feature_cols + ['trip_duration_minutes']].dropna()
if len(model_df) > MAX_ROWS:
    model_df = model_df.sample(n=MAX_ROWS, random_state=42)

print(f'Modellsorok száma (tanításhoz): {len(model_df):,}')
display(model_df.head())

Modellsorok száma (tanításhoz): 1,950,678


,passenger_count,trip_distance,pickup_lon,pickup_lat,dropoff_lon,dropoff_lat,pickup_hour,pickup_weekday,pickup_month,trip_duration_minutes
0,1.0,2.63,-73.991959,40.721565,-73.993805,40.695923,2.0,6.0,1.0,10.000000
1,3.0,4.55,-73.982101,40.736290,-73.955849,40.768028,3.0,6.0,1.0,7.000000
2,5.0,10.35,-74.002586,40.739746,-73.869980,40.770226,15.0,5.0,1.0,14.000000
3,1.0,5.00,-73.974266,40.790955,-73.996559,40.731850,20.0,3.0,1.0,21.033333
4,1.0,0.40,-74.001579,40.719383,-74.008377,40.720348,16.0,5.0,1.0,6.550000


## Baseline modell tanítása (OLS)

Itt két lineáris modellt tanítunk:
- `model_std`: skálázott OLS (összehasonlíthatóbb koefficiensekhez),
- `model_raw`: nyers OLS (mértékegységben értelmezhető koefficiensekhez).

In [14]:
X = model_df[feature_cols]
y = model_df['trip_duration_minutes']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model_std = Pipeline([
    ('scaler', StandardScaler()),
    ('reg', LinearRegression()),
])
model_std.fit(X_train, y_train)

model_raw = LinearRegression()
model_raw.fit(X_train, y_train)

pred_ols = model_std.predict(X_test)

print(f'OLS MAE  (perc): {mean_absolute_error(y_test, pred_ols):.3f}')
print(f'OLS RMSE (perc): {np.sqrt(mean_squared_error(y_test, pred_ols)):.3f}')
print(f'OLS R2         : {r2_score(y_test, pred_ols):.4f}')

OLS MAE  (perc): 3.507
OLS RMSE (perc): 6.002
OLS R2         : 0.4950


## 2. mérföldkő: hiperparaméter-választás és modell-összehasonlítás

`Ridge` és `Lasso` modelleknél az `alpha` szabályozza a regularizáció erősségét.
A legjobb `alpha` értéket `GridSearchCV`-vel, 3-szoros keresztvalidációval választjuk.

In [15]:
tune_rows = min(500_000, len(X_train))
X_train_tune = X_train.sample(n=tune_rows, random_state=42)
y_train_tune = y_train.loc[X_train_tune.index]

cv = KFold(n_splits=3, shuffle=True, random_state=42)
alpha_grid = {'reg__alpha': np.logspace(-3, 2, 8)}

ridge_pipe = Pipeline([('scaler', StandardScaler()), ('reg', Ridge())])
lasso_pipe = Pipeline([('scaler', StandardScaler()), ('reg', Lasso(max_iter=20_000))])

ridge_grid = GridSearchCV(ridge_pipe, alpha_grid, scoring='neg_mean_absolute_error', cv=cv, n_jobs=-1)
lasso_grid = GridSearchCV(lasso_pipe, alpha_grid, scoring='neg_mean_absolute_error', cv=cv, n_jobs=-1)

ridge_grid.fit(X_train_tune, y_train_tune)
lasso_grid.fit(X_train_tune, y_train_tune)

best_ridge = ridge_grid.best_estimator_
best_lasso = lasso_grid.best_estimator_

best_ridge.fit(X_train, y_train)
best_lasso.fit(X_train, y_train)

pred_ridge = best_ridge.predict(X_test)
pred_lasso = best_lasso.predict(X_test)

comparison_df = pd.DataFrame([
    {
        'model': 'LinearRegression (OLS)',
        'MAE': mean_absolute_error(y_test, pred_ols),
        'RMSE': np.sqrt(mean_squared_error(y_test, pred_ols)),
        'R2': r2_score(y_test, pred_ols),
    },
    {
        'model': 'Ridge (best alpha)',
        'MAE': mean_absolute_error(y_test, pred_ridge),
        'RMSE': np.sqrt(mean_squared_error(y_test, pred_ridge)),
        'R2': r2_score(y_test, pred_ridge),
    },
    {
        'model': 'Lasso (best alpha)',
        'MAE': mean_absolute_error(y_test, pred_lasso),
        'RMSE': np.sqrt(mean_squared_error(y_test, pred_lasso)),
        'R2': r2_score(y_test, pred_lasso),
    },
])

best_model_by_mae = comparison_df.loc[comparison_df['MAE'].idxmin(), 'model']
best_model_by_rmse = comparison_df.loc[comparison_df['RMSE'].idxmin(), 'model']
best_model_by_r2 = comparison_df.loc[comparison_df['R2'].idxmax(), 'model']

ranking_df = comparison_df.copy()
ranking_df['rank_mae'] = ranking_df['MAE'].rank(method='min', ascending=True)
ranking_df['rank_rmse'] = ranking_df['RMSE'].rank(method='min', ascending=True)
ranking_df['rank_r2'] = ranking_df['R2'].rank(method='min', ascending=False)
ranking_df['rank_mean'] = ranking_df[['rank_mae', 'rank_rmse', 'rank_r2']].mean(axis=1)
ranking_df = ranking_df.sort_values(['rank_mean', 'MAE']).reset_index(drop=True)

model_registry = {
    'LinearRegression (OLS)': model_std,
    'Ridge (best alpha)': best_ridge,
    'Lasso (best alpha)': best_lasso,
}

selected_model_name = ranking_df.loc[0, 'model']
selected_model = model_registry[selected_model_name]

print('Legjobb modell metrikánként:')
print(f'- MAE alapján : {best_model_by_mae}')
print(f'- RMSE alapján: {best_model_by_rmse}')
print(f'- R2 alapján  : {best_model_by_r2}')
print(f'Összesített legjobb modell: {selected_model_name}')

print('\nÖsszehasonlító tábla:')
display(comparison_df.sort_values('MAE').reset_index(drop=True))

print('Rangsor (összesített):')
display(ranking_df)

F:\Projects\interpretable-ml-taxi\.venv\Lib\site-packages\sklearn\linear_model\_ridge.py:228: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 5.002935288267452e-10.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
F:\Projects\interpretable-ml-taxi\.venv\Lib\site-packages\sklearn\linear_model\_ridge.py:228: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 1.602918492382699e-10.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T


Legjobb modell metrikánként:
- MAE alapján : Ridge (best alpha)
- RMSE alapján: Ridge (best alpha)
- R2 alapján  : Ridge (best alpha)
Összesített legjobb modell: Ridge (best alpha)

Összehasonlító tábla:


,model,MAE,RMSE,R2
0,Ridge (best alpha),3.506391,6.001449,0.495043
1,LinearRegression (OLS),3.507323,6.001848,0.494976
2,Lasso (best alpha),3.507383,6.001801,0.494984


Rangsor (összesített):


,model,MAE,RMSE,R2,rank_mae,rank_rmse,rank_r2,rank_mean
0,Ridge (best alpha),3.506391,6.001449,0.495043,1.0,1.0,1.0,1.000000
1,Lasso (best alpha),3.507383,6.001801,0.494984,3.0,2.0,2.0,2.333333
2,LinearRegression (OLS),3.507323,6.001848,0.494976,2.0,3.0,3.0,2.666667


## Interpretálhatóság

### 1) Koefficiensek (globális modellmagyarázat)
- `coef_raw_units`: nyers egységű hatás (mértékegységben értelmezhető),
- `coef_standardized`: skálázott hatás (feature-ök közötti összehasonlításhoz).

### 2) Permutation importance (modell-független)
A feature értékeinek összekeverésével mérjük, mennyit romlik a modell teljesítménye.
Minél nagyobb a romlás, annál fontosabb a feature.

In [16]:
coef_df = pd.DataFrame({
    'feature': feature_cols,
    'coef_raw_units': model_raw.coef_,
    'coef_standardized': model_std.named_steps['reg'].coef_,
})
coef_df['abs_coef_standardized'] = coef_df['coef_standardized'].abs()
coef_df = coef_df.sort_values('abs_coef_standardized', ascending=False)
display(coef_df)

,feature,coef_raw_units,coef_standardized,abs_coef_standardized
1,trip_distance,2.003505,5.948395,5.948395
6,pickup_hour,0.039375,0.238387,0.238387
7,pickup_weekday,0.031762,-0.142878,0.142878
2,pickup_lon,0.012933,0.061977,0.061977
3,pickup_lat,-0.006200,-0.060026,0.060026
4,dropoff_lon,0.008478,0.058069,0.058069
5,dropoff_lat,-0.003663,-0.056143,0.056143
0,passenger_count,0.013575,0.007290,0.007290
8,pickup_month,0.000000,0.000000,0.000000


In [17]:
perm = permutation_importance(
    selected_model,
    X_test,
    y_test,
    n_repeats=5,
    random_state=42,
    scoring='neg_mean_absolute_error',
)

perm_df = pd.DataFrame({
    'feature': feature_cols,
    'importance_mean': perm.importances_mean,
    'importance_std': perm.importances_std,
}).sort_values('importance_mean', ascending=False)

print(f'Permutation importance modell: {selected_model_name}')
display(perm_df)

Permutation importance modell: Ridge (best alpha)


,feature,importance_mean,importance_std
1,trip_distance,3.267167,0.005457
4,dropoff_lon,0.641578,0.001031
5,dropoff_lat,0.632079,0.001454
6,pickup_hour,0.012627,0.000438
7,pickup_weekday,0.003734,0.000176
3,pickup_lat,0.003221,0.000064
0,passenger_count,0.000049,0.000009
8,pickup_month,0.000000,0.000000
2,pickup_lon,-0.002236,0.000147


In [18]:
# Egy konkrét ETA példa: minden modell predikciója
example = X_test.iloc[[0]].copy()
actual_minutes = float(y_test.iloc[0])

rows = []
for model_name, model in model_registry.items():
    pred = float(model.predict(example)[0])
    rows.append({
        'model': model_name,
        'predicted_eta_min': pred,
        'actual_min': actual_minutes,
        'abs_error_min': abs(actual_minutes - pred),
    })

eta_comparison_df = pd.DataFrame(rows).sort_values('abs_error_min').reset_index(drop=True)
print('Példa bemenet:')
display(example)
print('ETA összehasonlítás ugyanarra a fuvarra:')
display(eta_comparison_df)

Példa bemenet:


,passenger_count,trip_distance,pickup_lon,pickup_lat,dropoff_lon,dropoff_lat,pickup_hour,pickup_weekday,pickup_month
1938711,2.0,1.6,-73.976944,40.758739,-73.991463,40.738735,20.0,4.0,1.0


ETA összehasonlítás ugyanarra a fuvarra:


,model,predicted_eta_min,actual_min,abs_error_min
0,LinearRegression (OLS),9.059334,7.766667,1.292667
1,Lasso (best alpha),9.059650,7.766667,1.292984
2,Ridge (best alpha),9.074903,7.766667,1.308237
